# Lesson 10 - Files, CSV, and JSON

**Goal:** Read and write structured records safely with pathlib, context managers, and explicit encodings.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Paths and temporary files

`Path` represents a filesystem path. Use UTF-8 explicitly. A `with` block closes an open file even if an exception occurs. Examples use a temporary directory that is deleted on exit, so rerunning does not alter your project files.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "note.txt"
    path.write_text("Salary review\nApproved\n", encoding="utf-8")
    with path.open(encoding="utf-8") as handle:
        lines = [line.strip() for line in handle]
    print(lines)
assert lines == ["Salary review", "Approved"]

['Salary review', 'Approved']


## 2. CSV records

Use the csv module instead of splitting on commas: quoted fields may contain commas. Open CSV with `newline=""`. DictReader returns strings, so validate and convert numeric fields after reading.

In [2]:
import csv

with TemporaryDirectory() as folder:
    path = Path(folder) / "employees.csv"
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "name", "basic"])
        writer.writeheader()
        writer.writerow({"id": 101, "name": "Sen, Asha", "basic": 52500})
    with path.open(newline="", encoding="utf-8") as handle:
        loaded_rows = list(csv.DictReader(handle))
    loaded_rows[0]["basic"] = float(loaded_rows[0]["basic"])
    print(loaded_rows)
assert loaded_rows[0]["name"] == "Sen, Asha"

[{'id': '101', 'name': 'Sen, Asha', 'basic': 52500.0}]


## 3. JSON and supported types

JSON stores objects, arrays, strings, numbers, booleans, and null. Python maps those to dictionaries, lists, strings, numbers, booleans, and None. Sets, Decimal, and dates need an explicit representation. Use string keys and allow_nan=False for strict numeric output.

In [3]:
import json

record = {"id": 101, "name": "Asha", "active": True, "increment_date": None}
encoded = json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False)
decoded = json.loads(encoded)
print(encoded)
assert decoded == record
print(json.loads(json.dumps((1, 2))))  # Tuple returns as a list.

{
  "id": 101,
  "name": "Asha",
  "active": true,
  "increment_date": null
}
[1, 2]


## 4. Validate the loaded shape

Valid JSON syntax does not imply valid employee data. Validate structure and required fields. Keep FileNotFoundError, JSONDecodeError, and invalid-record errors distinct.

In [4]:
def load_employee_text(text):
    record = json.loads(text)
    if not isinstance(record, dict):
        raise ValueError("Expected an employee object")
    if not {"id", "name"} <= record.keys():
        raise ValueError("Missing id or name")
    return record

print(load_employee_text('{"id": 101, "name": "Asha"}'))
for text in ['not json', '[]', '{"name": "Asha"}']:
    try:
        load_employee_text(text)
    except (json.JSONDecodeError, ValueError) as error:
        print(type(error).__name__, str(error))

{'id': 101, 'name': 'Asha'}
JSONDecodeError Expecting value: line 1 column 1 (char 0)
ValueError Expected an employee object
ValueError Missing id or name


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Text round trip

Write and read two UTF-8 employee names inside TemporaryDirectory.

**Check:** Read text matches the original including non-ASCII characters.

In [5]:
# Write your solution and checks here.

### Exercise 2. CSV numeric fields

Write three salary rows, read them using DictReader, convert basic pay, and total it.

**Check:** 52,500 + 75,000 + 112,600 = 240,100.

In [6]:
# Write your solution and checks here.

### Exercise 3. Quoted CSV

Round-trip a name containing a comma and a note containing a newline using csv.

**Check:** Both fields survive unchanged; do not split lines manually.

In [7]:
# Write your solution and checks here.

### Exercise 4. JSON schema checks

Require an employee ID, nonblank name, and nonnegative numeric salary after parsing JSON.

**Check:** Malformed JSON and valid JSON with invalid fields produce different messages.

In [8]:
# Write your solution and checks here.

### Exercise 5. Portable values

Encode an employee with a date string and a sorted list of departments instead of a set.

**Check:** The decoded object contains JSON-compatible types and predictable department order.

In [9]:
# Write your solution and checks here.

## Chapter project

Read a small CSV, keep accepted and rejected records with row numbers, and write a JSON report in a temporary directory. Reopen the report and verify counts and totals.

In [10]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `11_modules_and_project_structure.ipynb` - Modules and Project Structure.